<a href="https://colab.research.google.com/github/tttttt0608/-/blob/main/%E8%96%AF%E8%AF%84%E6%B4%9E%E5%AF%9Fdemo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install gradio -q

In [2]:
import gradio as gr
import re
from collections import Counter

In [3]:
CATEGORY_RULES = {
    "价格敏感": {
        "keywords": ["多少钱", "贵", "预算", "学生党", "平替", "便宜", "价格", "性价比", "划算"],
        "concern": "用户想降低试错成本，关心价格是否值得。",
        "cover": ["真实预算", "学生党必看", "别多花冤枉钱"]
    },
    "安全顾虑": {
        "keywords": ["安全吗", "安全", "女生", "一个人", "独居", "晚上", "危险", "治安"],
        "concern": "用户担心现实风险，尤其关注个人安全和生活环境。",
        "cover": ["女生必看", "一个人也要安全", "安全避坑"]
    },
    "交易风险": {
        "keywords": ["押金", "合同", "中介", "被骗", "退款", "定金", "签约", "违约"],
        "concern": "用户担心交易流程不透明，害怕踩坑或产生损失。",
        "cover": ["押金别乱交", "合同避坑", "中介费看清楚"]
    },
    "地点选择": {
        "keywords": ["哪里", "附近", "区域", "推荐", "位置", "交通", "地铁", "学校", "NUS", "NTU"],
        "concern": "用户需要更具体、可执行的地点建议。",
        "cover": ["区域怎么选", "新手地点指南", "附近推荐"]
    },
    "适配疑问": {
        "keywords": ["适合吗", "能用吗", "会不会", "可以吗", "适合", "新手", "敏感肌", "油皮", "黄黑皮"],
        "concern": "用户想知道内容是否适合自己的具体情况。",
        "cover": ["到底适合谁", "新手先看", "真实测评"]
    }
}

In [4]:
SAMPLE_COMMENTS = """姐妹这个适合预算低的学生党吗？
女生一个人租房安全吗？
押金一般多久能退？
NUS 附近有什么区域推荐吗？
中介费是不是很坑？
可以出一期租房合同避坑吗？
一个人晚上回家会不会不安全？
有没有适合女生的区域？"""

In [5]:
def split_comments(text):
    lines = re.split(r"\n|。|？|\?|！|!", text)
    return [line.strip() for line in lines if line.strip()]


In [16]:
def analyze_comments(category, topic, goal, comments):
    if not comments.strip():
        return "请先粘贴评论区内容。"

    lines = split_comments(comments)
    total = len(lines)

    results = []

    for name, rule in CATEGORY_RULES.items():
        matched_comments = []
        matched_keywords = []

        for line in lines:
            for keyword in rule["keywords"]:
                if keyword in line:
                    matched_comments.append(line)
                    matched_keywords.append(keyword)
                    break

        if matched_comments:
            results.append({
                "name": name,
                "count": len(matched_comments),
                "concern": rule["concern"],
                "keywords": list(dict.fromkeys(matched_keywords)),
                "comments": matched_comments,
                "cover": rule["cover"]
            })

    if not results:
        return f"""
# 分析结果

暂时没有识别到明显的需求类型。

建议你粘贴更多评论，或者加入更具体的问题表达，例如：

- 多少钱
- 适合吗
- 安全吗
- 哪里买
- 有平替吗
- 可以出一期吗
"""

    results = sorted(results, key=lambda x: x["count"], reverse=True)
    top = results[0]

    opportunity_score = min(10, round((top["count"] / total) * 12 + 5))

    topics = generate_topics(topic, top["name"], goal)
    reply = generate_reply(topic, top["name"])

    demand_table = "| 需求类型 | 命中次数 | 关键词 | 用户真实关心 |\n"
    demand_table += "|---|---:|---|---|\n"

    for item in results:
        demand_table += f"| {item['name']} | {item['count']} | {'、'.join(item['keywords'])} | {item['concern']} |\n"

    typical_comments = ""
    for item in results[:3]:
        typical_comments += f"\n### {item['name']}\n"
        for c in item["comments"][:3]:
            typical_comments += f"- {c}\n"

    topic_list = ""
    for i, t in enumerate(topics, 1):
        topic_list += f"{i}. {t}\n"

    cover_text = " / ".join(top["cover"])

    output = f"""
# 薯评洞察分析结果

## 1. 核心洞察

本次共识别 **{total} 条评论**。

评论区最突出的用户需求是：**{top['name']}**。

**需求解释：**
{top['concern']}

**内容机会评分：{opportunity_score}/10**

这个分数基于该类评论出现频率，以及它是否适合转化为收藏型、评论型或 FAQ 型内容。

---

## 2. 用户需求分类

{demand_table}

---

## 3. 典型评论

{typical_comments}

---

## 4. 下一篇笔记选题建议

{topic_list}

---

## 5. 推荐标题

**{topics[0]}**

---

## 6. 封面文案建议

**{cover_text}**

---

## 7. 评论回复模板

{reply}

---

## 8. 产品提示

当前版本没有连接真实小红书数据，而是使用手动粘贴评论的方式进行分析。这样可以避免数据权限和隐私问题。后续版本可以接入创作者授权数据，并加入更强的语义分析能力。
"""

    return output

In [21]:
def generate_topics(topic, category_name, goal):
    base = topic if topic else "这个主题"

    templates = {
        "价格敏感": [
            f"{base}真实预算整理：学生党到底要准备多少钱？",
            f"{base}省钱避坑指南：哪些钱可以不花？",
            f"第一次做{base}，我建议先看清这些隐藏成本"
        ],
        "安全顾虑": [
            f"女生一个人做{base}，最该注意的不是价格",
            f"{base}安全指南：新手一定要提前确认的细节",
            f"一个人也能安心的{base}经验分享"
        ],
        "交易风险": [
            f"{base}避坑：押金、合同和中介这些点别乱签",
            f"{base}前一定要问清楚的 7 个问题",
            f"我踩过的{base}交易坑，建议你提前知道"
        ],
        "地点选择": [
            f"{base}区域怎么选？新手友好版整理",
            f"{base}地点推荐：适合不同预算的人怎么选",
            f"{base}附近怎么选？交通、价格和便利度对比"
        ],
        "适配疑问": [
            f"{base}到底适合谁？不适合谁？",
            f"新手做{base}前，先看这一篇真实经验",
            f"{base}常见问题整理：评论区问最多的都在这里"
        ]
    }

    result = templates.get(category_name, [
        f"{base}评论区高频问题整理",
        f"{base}新手避坑指南",
        f"{base}真实经验分享"
    ])

    if goal == "提高收藏":
        result = [item + "｜建议收藏版" for item in result]
    elif goal == "提高评论":
        result = [item + "｜评论区答疑版" for item in result]
    elif goal == "做FAQ内容":
        result = [item + "｜FAQ整理版" for item in result]

    return result

In [22]:
def generate_reply(topic, category_name):
    base = topic if topic else "这个主题"

    replies = {
        "价格敏感": f"可以的，我下一篇会整理{base}的真实预算，包括学生党比较关心的价格、隐藏成本和省钱建议。",
        "安全顾虑": f"这个问题很重要。我会单独出一期{base}安全注意事项，尤其会补充女生一个人需要提前确认的细节。",
        "交易风险": f"可以，我下一篇会重点写{base}里的押金、合同和中介问题，把容易踩坑的地方列清楚。",
        "地点选择": f"收到，我会整理一期{base}地点选择指南，按照交通、预算和便利度来对比。",
        "适配疑问": f"这个我可以单独讲一下。下一篇会整理{base}到底适合哪些人，以及哪些情况需要谨慎。"
    }

    return replies.get(category_name, "收到，我会把评论区问得最多的问题整理成下一篇笔记。")


with gr.Blocks(theme=gr.themes.Soft(primary_hue="red", neutral_hue="slate")) as demo:
    gr.Markdown(
        """
# 薯评洞察｜小红书评论区需求挖掘助手

把小红书评论粘贴进来，系统会自动识别用户需求，并生成下一篇笔记选题、标题封面和评论回复模板。

**Demo 说明：** 当前版本不爬取真实小红书数据，只分析用户手动粘贴的评论，适合作为 48 小时产品 Demo。
"""
    )

    with gr.Row():
        category = gr.Dropdown(
            choices=["美妆", "穿搭", "留学", "求职", "探店", "母婴", "生活方式"],
            value="留学",
            label="内容领域"
        )
        topic = gr.Textbox(
            value="新加坡租房避坑",
            label="当前笔记主题"
        )
        goal = gr.Dropdown(
            choices=["找下一篇选题", "提高收藏", "提高评论", "促进转化", "做FAQ内容"],
            value="找下一篇选题",
            label="创作目标"
        )

    comments = gr.Textbox(
        value=SAMPLE_COMMENTS,
        label="评论区内容",
        lines=10,
        placeholder="把评论一行一条粘贴在这里"
    )

    analyze_btn = gr.Button("生成评论洞察", variant="primary")

    output = gr.Markdown(label="分析结果")

    analyze_btn.click(
        fn=analyze_comments,
        inputs=[category, topic, goal, comments],
        outputs=output
    )

demo.launch(share=True)

/tmp/ipykernel_14698/1929970363.py:15: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft(primary_hue="red", neutral_hue="slate")) as demo:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://ef2efb1c76797a84d0.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
